## Load Dataset

In [ ]:
#importing required libraries
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
#Load dataset from GitHub
df = pd.read_csv("https://raw.githubusercontent.com/suryautharakumar/Technology-Dissertation/refs/heads/main/Fuel-Price.csv")

print("Dataset loaded successfully!\n")
print("Shape of dataset:", df.shape)

Dataset loaded successfully!

Shape of dataset: (41406, 6)


## Dataset Understanding

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41406 entries, 0 to 41405
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   city_name   41406 non-null  object 
 1   state_name  41406 non-null  object 
 2   date        41406 non-null  object 
 3   petrol      41398 non-null  float64
 4   diesel      41398 non-null  float64
 5   xpremium    41398 non-null  float64
dtypes: float64(3), object(3)
memory usage: 1.9+ MB


In [ ]:
df.head()

,city_name,state_name,date,petrol,diesel,xpremium
0,Nicobar,Andaman and Nicobar Islands,09-07-24,82.42,78.01,85.42
1,Nicobar,Andaman and Nicobar Islands,08-07-24,82.42,78.01,85.42
2,Nicobar,Andaman and Nicobar Islands,07-07-24,82.42,78.01,85.42
3,Nicobar,Andaman and Nicobar Islands,06-07-24,82.42,78.01,85.42
4,Nicobar,Andaman and Nicobar Islands,05-07-24,82.42,78.01,85.42


In [ ]:
#Including categorical columns
df.describe(include='all')

,city_name,state_name,date,petrol,diesel,xpremium
count,41406,41406,41406,41398.000000,41398.000000,41398.000000
unique,692,32,69,NaN,NaN,NaN
top,Bilaspur,Uttar Pradesh,09-06-24,NaN,NaN,NaN
freq,130,3525,695,NaN,NaN,NaN
mean,NaN,NaN,NaN,101.041052,90.824937,104.101261
std,NaN,NaN,NaN,5.246480,3.933354,5.262601
min,NaN,NaN,NaN,82.420000,78.010000,85.420000
25%,NaN,NaN,NaN,96.040000,88.140000,99.040000
50%,NaN,NaN,NaN,101.170000,91.410000,104.260000
75%,NaN,NaN,NaN,105.980000,93.650000,108.950000


In [ ]:
df.isnull().sum()

,0
city_name,0
state_name,0
date,0
petrol,8
diesel,8
xpremium,8


In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [ ]:
#Check unique values
df.nunique()

,0
city_name,692
state_name,32
date,69
petrol,641
diesel,642
xpremium,684


## Define Target and Features

In [ ]:
df = df.dropna(subset=['petrol']).copy()

print("Shape after removing missing target values:", df.shape)
print("Missing petrol values:", df['petrol'].isnull().sum())

Shape after removing missing target values: (41398, 6)
Missing petrol values: 0


In [ ]:
target = 'petrol'

X = df.drop(target, axis=1)
y = df[target]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (41398, 5)
Target: (41398,)


In [ ]:
numeric_features = X.select_dtypes(
    include=['int64', 'float64']
).columns

categorical_features = X.select_dtypes(
    include=['object']
).columns

print("Numerical features:")
print(list(numeric_features))

print("\nCategorical features:")
print(list(categorical_features))

Numerical features:
['diesel', 'xpremium']

Categorical features:
['city_name', 'state_name', 'date']


## Data Cleaning

In [ ]:
num_imputer = SimpleImputer(strategy='median')

X[numeric_features] = num_imputer.fit_transform(
    X[numeric_features]
)

In [ ]:
cat_imputer = SimpleImputer(strategy='most_frequent')

X[categorical_features] = cat_imputer.fit_transform(
    X[categorical_features]
)

In [ ]:
#Checking missing values

print("Total missing values:", X.isnull().sum().sum())

Total missing values: 0


## Process Date

In [ ]:
#Convert date to datetime

X['date'] = pd.to_datetime(X['date'])

/tmp/ipykernel_3113/1553707131.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X['date'] = pd.to_datetime(X['date'])


In [ ]:
#Extract date
X['year'] = X['date'].dt.year
X['month'] = X['date'].dt.month
X['day'] = X['date'].dt.day
X['day_of_week'] = X['date'].dt.dayofweek

In [ ]:
#Remove original date column

X = X.drop('date', axis=1)

In [ ]:
#One-Hot Encoding

X = pd.get_dummies(
    X,
    drop_first=True
)

print("Shape after encoding:", X.shape)

Shape after encoding: (41398, 728)


In [ ]:
#Feature Scaling
scaler = StandardScaler()

X = scaler.fit_transform(X)

print("Feature scaling completed.")

Feature scaling completed.


In [ ]:
#Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (33118, 728)
Testing data: (8280, 728)


## Model Building

In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree Regressor": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest Regressor": RandomForestRegressor(
        random_state=42
    ),

    "Gradient Boosting Regressor": GradientBoostingRegressor(
        random_state=42
    )
}

In [ ]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )

    r2 = r2_score(y_test, y_pred)

    results.append([
        name,
        mae,
        rmse,
        r2
    ])

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R² Score"
    ]
)

results_df = results_df.sort_values(
    by="R² Score",
    ascending=False
).reset_index(drop=True)

results_df

,Model,MAE,RMSE,R² Score
0,Decision Tree Regressor,0.000012,0.000659,1.000000
1,Random Forest Regressor,0.000049,0.000881,1.000000
2,Linear Regression,0.007922,0.034233,0.999957
3,Gradient Boosting Regressor,0.158034,0.210550,0.998368


In [ ]:
#best-performing model

best_model = results_df.iloc[0]

print("Best Performing Model")
print("----------------------")
print("Model:", best_model["Model"])
print("MAE:", round(best_model["MAE"], 4))
print("RMSE:", round(best_model["RMSE"], 4))
print("R² Score:", round(best_model["R² Score"], 4))

Best Performing Model
----------------------
Model: Decision Tree Regressor
MAE: 0.0
RMSE: 0.0007
R² Score: 1.0
